# Évaluation Ragas de la chaîne RAG — jeu de test annoté

Charge les résultats déjà calculés par `eval/evaluate_rag.py` (`eval/eval_results.json`) sur les questions annotées de `eval/qa_dataset_manual.json` — toutes écrites et vérifiées à la main contre `data/processed/events.json` (voir `eval/generate_testset.py` pour l'alternative Ragas explorée puis abandonnée, staleness et doublons non fiables) — plutôt que de relancer les appels Mistral/Ragas en direct dans ce notebook, pour disposer d'une trace stable et reproductible à présenter, sans dépendance réseau ni coût API le jour de la soutenance.

Métriques (0 à 1, plus haut = meilleur) :
- **faithfulness** — la réponse s'appuie-t-elle uniquement sur le contexte récupéré, sans invention ?
- **answer_relevancy** — la réponse répond-elle réellement à la question posée ?
- **context_recall** — le contexte récupéré contient-il toute l'information nécessaire pour bien répondre ?
- **answer_correctness** — la réponse générée correspond-elle à la réponse de référence (ground truth) ?

Les deux premières jugent la chaîne RAG dans l'absolu ; les deux dernières la comparent au jeu de test annoté.

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", None)

RESULTS_PATH = Path("..") / "eval" / "eval_results.json"
results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
df = pd.DataFrame(results)
print(f"{len(df)} questions évaluées.")

## Scores moyens

In [ ]:
for metric in ["faithfulness", "answer_relevancy", "context_recall", "answer_correctness"]:
    print(f"{metric:<20} moyenne : {df[metric].mean():.2f}")

## Détail par question

Triées par faithfulness croissante — les cas les plus fragiles en premier.

In [ ]:
df.sort_values("faithfulness")[["question", "dimension", "faithfulness", "answer_relevancy", "context_recall", "answer_correctness"]]

## Questions avec un score de fidélité faible (< 0.7)

Un score de faithfulness bas signifie que la réponse générée contient des affirmations non étayées par le contexte récupéré — cas à examiner en priorité, potentiel signe d'hallucination.

In [ ]:
low_faithfulness = df[df["faithfulness"] < 0.7]

if low_faithfulness.empty:
    print("Aucune question sous ce seuil.")
else:
    for _, row in low_faithfulness.iterrows():
        print(f"### {row['question']}")
        print(f"Faithfulness : {row['faithfulness']:.2f}")
        print(f"Réponse générée : {row['generated_answer']}")
        print(f"Réponse de référence : {row['reference_answer']}")
        print()

## Réponses générées vs réponses de référence, question par question

In [ ]:
df[["question", "dimension", "generated_answer", "reference_answer", "faithfulness", "answer_relevancy", "context_recall", "answer_correctness"]]